## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-15-02-53-00 +0000,wsj,Hear How the FAA’s Ground Stop Unfolded,https://www.wsj.com/business/airlines/hear-how...
1,2026-11-15-02-53-00 +0000,wsj,A federal judge temporarily barred the Trump a...,https://www.wsj.com/politics/policy/trump-admi...
2,2026-11-15-02-52-12 +0000,nyt,How Some Flights Were Exempt From the Ground Stop,https://www.nytimes.com/2026/11/14/business/fa...
3,2026-11-15-02-42-57 +0000,nyt,Judge Blocks Immediate Closure of East Potomac...,https://www.nytimes.com/2026/11/14/us/politics...
4,2026-11-15-02-41-35 +0000,startribune,"Live: Gophers, after travel issues, trail late...",https://www.startribune.com/minnesota-gophers-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2622/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
2,ground,20
93,california,16
7,trump,16
130,first,14
63,new,13


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
138,2026-11-14-15-53-55 +0000,bbc,"Trump, Putin and Xi plan historic joint meetin...",https://www.bbc.co.uk/news/articles/ckz74jzpy1...,62
163,2026-11-14-13-10-00 +0000,wsj,Weekend Air Travel Woes Mount After FAA Lifts ...,https://www.wsj.com/us-news/faa-temporarily-ha...,62
94,2026-11-14-20-08-07 +0000,cbc,New Orleans port sues for control of China-lin...,https://www.cbc.ca/news/world/new-orleans-tank...,53
93,2026-11-14-20-10-01 +0000,nyt,‘Democrat’ in Alaska’s House Race Is Doing Tim...,https://www.nytimes.com/2026/11/14/us/election...,52
73,2026-11-14-21-37-22 +0000,nypost,Travelers still scrambling at California airpo...,https://nypost.com/2026/11/14/us-news/traveler...,51


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
138,62,2026-11-14-15-53-55 +0000,bbc,"Trump, Putin and Xi plan historic joint meetin...",https://www.bbc.co.uk/news/articles/ckz74jzpy1...
163,62,2026-11-14-13-10-00 +0000,wsj,Weekend Air Travel Woes Mount After FAA Lifts ...,https://www.wsj.com/us-news/faa-temporarily-ha...
18,48,2026-11-15-01-02-12 +0000,nypost,Tanker that struck New Orleans cruise ship had...,https://nypost.com/2026/11/14/us-news/tanker-t...
213,47,2026-11-14-05-08-59 +0000,nyt,A Democrat Wins the Alaska Governor’s Race for...,https://www.nytimes.com/2026/11/14/us/election...
126,46,2026-11-14-17-24-30 +0000,nypost,How ex-Nvidia CEO Jensen Huang threw tantrums ...,https://nypost.com/2026/11/14/business/how-ex-...
208,41,2026-11-14-06-42-27 +0000,bbc,Microsoft says services fully restored after d...,https://www.bbc.co.uk/news/articles/c6rrg51rkz...
47,40,2026-11-14-23-05-36 +0000,nypost,Raunchy posts from upstart Dem who took down C...,https://nypost.com/2026/11/14/us-news/past-twe...
113,31,2026-11-14-18-35-00 +0000,wsj,Transportation Secretary Sean Duffy blames Con...,https://www.wsj.com/politics/policy/duffy-blam...
211,29,2026-11-14-05-50-15 +0000,nypost,Dems launch sexual harassment probe against ne...,https://nypost.com/2026/11/14/us-news/dems-lau...
1,26,2026-11-15-02-53-00 +0000,wsj,A federal judge temporarily barred the Trump a...,https://www.wsj.com/politics/policy/trump-admi...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
